# Practical Block III: Grand Challenge
# BACH: Grand Challenge on Breast Cancer Histology Images
By: Niklas Long Schiefelbein

---

## General Challenge Information

This Jupyter Notebook is presenting the results and findings of the third and last exercise of the course DLMIA based on the BACH challenge. This exercise did only consider Part A of the challenge - the classification of H&E stained breast histology microscopy images in four classes: *Normal*, *Benign*, *In situ carcinoma* and *Invasive carcinoma*.

The dataset for this part of the challenge consists of 400 training images (originally there was another test set consisting of 100 images to which no access was available), in which the four classes are equally represented. The images were acquired in 2014, 2015 and 2017 using a Leica DM 2000 LED microscope and a Leica ICC50 HD camera and all patients are from the Porto and Castelo Branco regions (Portugal).

---

## Methodology and Approach in this Exercise

The overall approach consisted of orienting around what the challenge winner of Part A [(Chennamsetty et al.)](https://link.springer.com/chapter/10.1007/978-3-319-93000-8_91) did. Particularly, they decided to implement a classification ensemble consisting of one ResNet101 model and two DenseNet161 models of which the latter two incorporate different normalisation schemes (one is normalised on ImageNet and the other on the BACH dataset). However, simply clining their approach, which lead them win this challenge, does not grant the pedagogical benefits that could be acquired otherwise. Instead, the exercise's goal consisted of analysing which data handling techniques affect the performance and how. With this objective in mind, a multi-step approach was choosen in which several configurations are tested stage-wise:

1. **Data Augmentation:** Firstly, the effect of augmentations were analysed. Four different sets of augmentation were developed in order to assess their effect on the classification performance. 
    - **No Augmentation:** This only includes resizing and normalisation which are necessary steps in order to alter the input samples to work with CNNs. This scheme is also used for validation and testing since those steps have to be performed on the original data but resized and normalised.
    - **Simple Geometric Augmentations:** Includes horizontal and vertical flipping and random rotation. Those augmentation techniques imitate different orientations when taking photos of the tissue under the microscope. Since there is no inherent information in how a microscopy image is oriented, those augmentations can be used easily in order to increase the variance of the dataset.
    - **Complex Geometric Augmentations:** Includes all of the above augmentations but extends the set by `ShiftScaleRotate` and `ElasticTransform`. The first is again a more aggressive form of the already implemented augmentations whereas the latter simlulates differently distributed stretches of tissue.
    - **Photometric Augmentations:** Again, this set includes all of the above methods but extends with `HueSaturationValue` and `RandomBrightnessContrast` to account for variability in the colourspace. This should simulate different stain manufacturers, “fresh” stains and older, more faded slides and scanner light intensities.

    At first, I wanted to only include geometric augmentations since I am missing the knowledge to assess if certain critical information is lost when altering the colour space but following [TODO](https://ajkfja.de) I decided to include `HueSaturationValue` and `RandomBrightnessContrast` with predefined boundaries according to the paper’s findings.

2. **Image Resolution:** The second analysis consisted of analysing the effect of doubling the input width and height each. Instead of 256x256, the images were resized to 512x512 while reducing the batch size from 32 to 8 in order to account for the increased GPU capacity utilisation. 

3. **Learning Rate:** The third stage of the analysis takes the best performing configurations and runs the training once again but exploring different learning rates: [0.0001, 0.0005, 0.001].

4. **Ensemble Run:** Lastly, an ensemble run is conducted which combines a 5-fold cross validation of each model's prediction and voting to get a more reliable and model-independent classification. Important to mention is that here, due to the integrated cross validation, it was chosen to apply a soft voting mechanism instead of a majority voting which was used in the original paper. The implemented system is demonstrated in Figure 1.

<img src="soft_voting_cross_val.png" alt="grand_ensemble_system" width="600"/>

<small>Figure 1: The implemented Grand Ensemble System including Cross Validation and Soft Voting</small>
<br><small>Note: This diagram was generated by Gemini 3 based on what has been implemented.</small>

# TODO Still to mention:
- split structure
- absence of different augmentation techniques due to fear of corrupting the samples
- stratified split
- soft voting with bagging
- mention that reached performance is not as high as challenge winners but system is more robust towards differences in staining and imaging
- mention that benign was the worst in the challenge which my ensemble was able to classify to 100% correctly
- mention that little data


---
## Results and Findings during this Exercise

### Preliminary Experiments: 1. Augmentation

<img src="summary_accuracy_grid.png" alt="Accuracy Histories with different Augmentation Schemes" width="100%"/>

<small>Figure 2: Overview of Accuracy Histories for each Model for each Augmentation Scheme</small>

<img src="summary_loss_grid.png" alt="Loss Histories with different Augmentation Schemes" width="100%"/>

<small>Figure 3: Overview of Loss Histories for each Model for each Augmentation Scheme</small>

Figure 2 and figure 3 showcase the evolution of training and loss histories throughout applying different augmentation sets. What is evident and what was most influential towards the analysis is the clear sign of overfitting in early stages when no augmentation is applied. All models were able to reach 95% training accuracy or higher already after the second epoch while the validation accuracy stagnates. Since balancing overfitting and underfitting is one of the biggest challenges of deep learning, and already considering the quite high performances in accuracy of the very first baselines, the common thread throughout this practical was significantly aligned towards increasing the training complexity (regularising) to mitigate overfitting and improve generalisation.

The increasing complexity of augmentations that are applied lead to less steep learning curves which signalise the effect of an increasing difficulty to just memorise the data. All trainings show signs of increased volatility, which could have been mitigated with longer training (but was chosen not to due to limited computational ressources).

Just by looking at the graphs and validation performances of the last epoch, one can determine Augmentation Scheme 2 to yield the best results for each model. However, even though I was hesitant towards applying photometric augmentations first, I decided to use Scheme number 3 in further experiments to incorporate the (theoretical) variability of different scanning conditions.